In [1]:
import re
import string
import joblib
import nltk
import warnings

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

warnings.filterwarnings("ignore")

In [2]:
model = joblib.load("../models/phishing_detector.pkl")

vectorizer = joblib.load("../models/tfidf_vectorizer.pkl")

In [3]:
lemmatizer = WordNetLemmatizer()

stop_words = set(stopwords.words("english"))

In [4]:
def preprocess_text(text):

    text = text.lower()

    text = re.sub(r"<.*?>", "", text)

    text = re.sub(r"http\S+|www\S+", "", text)

    text = re.sub(r"\S+@\S+", "", text)

    text = re.sub(r"\d+", "", text)

    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )

    tokens = nltk.word_tokenize(text)

    tokens = [
        word
        for word in tokens
        if word not in stop_words
    ]

    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]

    return " ".join(tokens)

In [5]:
from scipy.sparse import hstack
import numpy as np

def extract_features(message):
    cleaned = preprocess_text(message)

    text_features = vectorizer.transform([cleaned])

    metadata = np.array([[
        len(message),
        len(re.findall(r"http|www", message)),
        len(re.findall(r"\d", message)),
        message.count("!"),
        sum(1 for c in message if c.isupper())
    ]])

    return hstack([text_features, metadata])

In [6]:
def predict_message(message):

    features = extract_features(message)

    prediction = model.predict(features)[0]

    if prediction == 1:
        return "🚨 SPAM"
    else:
        return "✅ HAM"

In [7]:
sample = "Congratulations! You have won a FREE iPhone. Click here to claim your prize."

print("Message:")
print(sample)

print("\nPrediction:")
print(predict_message(sample))

Message:
Congratulations! You have won a FREE iPhone. Click here to claim your prize.

Prediction:
🚨 SPAM


In [8]:
samples = [

    "Hey, are we still meeting at 5 PM?",

    "URGENT! Your account has been suspended. Verify immediately.",

    "Congratulations! Claim your FREE vacation now!",

    "Don't forget to bring your notebook tomorrow.",

    "Win ₹50,000 today by clicking this exclusive link!"
]

for msg in samples:

    print("=" * 70)

    print(msg)

    print()

    print("Prediction:", predict_message(msg))

Hey, are we still meeting at 5 PM?

Prediction: ✅ HAM
URGENT! Your account has been suspended. Verify immediately.

Prediction: ✅ HAM
Congratulations! Claim your FREE vacation now!

Prediction: 🚨 SPAM
Don't forget to bring your notebook tomorrow.

Prediction: ✅ HAM
Win ₹50,000 today by clicking this exclusive link!

Prediction: 🚨 SPAM


In [9]:
user_message = input("Enter a message: ")

print()

print("Prediction:")

print(predict_message(user_message))


Prediction:
✅ HAM


In [10]:
def predict_with_confidence(message):

    features = extract_features(message)

    prediction = model.predict(features)[0]

    probability = model.predict_proba(features).max()

    label = "SPAM" if prediction == 1 else "HAM"

    return label, probability

In [11]:
label, confidence = predict_with_confidence(sample)

print(f"Prediction : {label}")
print(f"Confidence : {confidence:.2%}")

Prediction : SPAM
Confidence : 87.65%


In [12]:
print("=" * 60)
print("Prediction System Ready")
print("=" * 60)

Prediction System Ready
